# 03 · Join Sofascore + Capology — Germany Bundesliga 23/24

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2023/24 de Bundesliga alemana**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_germany_2324.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_germany_2324.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  494 jugadores | 116 columnas
Capology:   557 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   1 fc heidenheim
   1 fc koln
   1 fc union berlin
   1 fsv mainz 05
   bayer 04 leverkusen
   borussia m gladbach
   darmstadt 98
   fc augsburg
   fc bayern munchen
   rb leipzig
   sc freiburg
   sv werder bremen
   tsg hoffenheim
   vfb stuttgart
   vfl bochum 1848
   vfl wolfsburg

En Capology pero no en Sofascore:
   augsburg
   bayer leverkusen
   bayern munich
   bochum
   darmstadt
   freiburg
   heidenheim
   hoffenheim
   koln
   leipzig
   mainz
   monchengladbach
   stuttgart
   union berlin
   werder bremen
   wolfsburg


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [6]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'augsburg':'fc augsburg',
            'bayer leverkusen':'bayer 04 leverkusen',
            'bayern munich':'fc bayern munchen',
            'bochum':'vfl bochum 1848',
            'darmstadt':'darmstadt 98',
            'freiburg':'sc freiburg',
            'heidenheim':'1 fc heidenheim',
            'hoffenheim':'tsg hoffenheim',
            'koln':'1 fc koln',
            'leipzig':'rb leipzig',
            'mainz':'1 fsv mainz 05',
            'monchengladbach':'borussia m gladbach',
            'stuttgart':'vfb stuttgart',
            'union berlin':'1 fc union berlin',
            'werder bremen':'sv werder bremen',
            'wolfsburg':'vfl wolfsburg'                          
}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')


✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [7]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 446/494 (90.3%)
Sin emparejar: 48


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [8]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          3
Revisión media    (0.75 ≤ score < 0.90):   8
Revisión estricta (0.50 ≤ score < 0.75):   24
Revisión muy est. (score < 0.50):           13


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [9]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
2,Frederik Rønnow,1. FC Union Berlin,frederik ronnow,0.966
30,Stanley N'Soki,TSG Hoffenheim,stanley nsoki,0.963
9,Joakim Mæhle,VfL Wolfsburg,joakim maehle,0.917


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [10]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
23,Joseph Scally,Borussia M'gladbach,joe scally,0.870
1,Victor Okoh Boniface,Bayer 04 Leverkusen,victor boniface,0.857
10,Jamie Gittens,Borussia Dortmund,jamie bynoe gittens,0.812
5,Leandro Barreiro,1. FSV Mainz 05,leandro barreiro martins,0.800
35,Matondo-Merveille Papela,1. FSV Mainz 05,merveille papela,0.800
6,Mateu Morey,Borussia Dortmund,mateu morey bauza,0.786
21,Dion Drena Beljo,FC Augsburg,dion beljo,0.769
24,Haktab Omar Traore,1. FC Heidenheim,omar traore,0.759


In [11]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')


Aceptados: 8 | Excluidos: 0


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [12]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
14,John Brooks,TSG Hoffenheim,john anthony brooks,0.733
37,Maximilian Breunig,SC Freiburg,maximilian eggestein,0.684
4,Jeff Chabot,1. FC Köln,julian chabot,0.667
40,Oliver Burke,SV Werder Bremen,olivier deman,0.640
0,Miloš Pantović,1. FC Union Berlin,josip juranovic,0.621
8,Junior Dina Ebimbe,Eintracht Frankfurt,eric ebimbe,0.621
13,Silas,VfB Stuttgart,gil dias,0.615
15,Rafael Borré,SV Werder Bremen,santos borre,0.583
12,Manu Koné,Borussia M'gladbach,kouadio kone,0.571
3,Kjell Wätjen,Borussia Dortmund,donyell malen,0.560


In [13]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['john brooks',
                    'jeff chabot',
                    'junior dina ebimbe',
                    'rafael borre',
                    'manu kone'

]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')


Aceptados del nivel bajo: 5


### 7.4 Revisión muy estricta (score < 0.50)

Por defecto ninguno se acepta. Añadir a `ACCEPT_VERY_LOW_FUZZY` los correctos.

In [14]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
46,Joel Imasuen,SV Werder Bremen,olivier deman,0.480
38,Asaf Arania,Darmstadt 98,bartol franjic,0.480
18,Ryan Gravenberch,FC Bayern München,bryan zaragoza,0.467
32,Thorgan Hazard,Borussia Dortmund,mateu morey bauza,0.452
29,Randal Kolo Muani,Eintracht Frankfurt,sasa kalajdzic,0.452
25,Tim Drexler,TSG Hoffenheim,maximilian beier,0.444
31,Jesper Lindstrøm,Eintracht Frankfurt,jens petter hauge,0.438
44,Nuha Jatta,RB Leipzig,amadou haidara,0.417
45,Mahmut Kücüksahin,FC Augsburg,tomas koubek,0.414
42,Jonathan Asp Jensen,FC Bayern München,harry kane,0.414


In [15]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')


Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [16]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 462/494 (93.5%)
Sin salario:     32


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [17]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 32


,player,team,minutesPlayed,appearances,goals,assists
0,Miloš Pantović,1. FC Union Berlin,14,1,1,0
1,Anwar El Ghazi,1. FSV Mainz 05,51,3,0,1
2,Marcus Müller,1. FSV Mainz 05,17,1,0,0
3,Kjell Wätjen,Borussia Dortmund,104,2,0,1
4,Samuel Bamba,Borussia Dortmund,45,2,0,0
5,Hendry Blank,Borussia Dortmund,45,1,0,0
6,Thorgan Hazard,Borussia Dortmund,13,1,0,0
7,Asaf Arania,Darmstadt 98,10,1,0,0
8,Randal Kolo Muani,Eintracht Frankfurt,155,2,1,0
9,Jesper Lindstrøm,Eintracht Frankfurt,84,1,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [18]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  1. FC Union Berlin  —  SF sin salario:


,player,minutesPlayed
0,Miloš Pantović,14


  CG plantilla completa:


,player,player_norm
0,Aïssa Laïdouni,aissa laidouni
1,Alex Kral,alex kral
2,Alexander Schwolow,alexander schwolow
3,Aljoscha Kemlein,aljoscha kemlein
4,András Schäfer,andras schafer
5,Benedict Hollerbach,benedict hollerbach
6,Brenden Aaronson,brenden aaronson
7,Chris Bedia,chris bedia
8,Christopher Trimmel,christopher trimmel
9,Danilho Doekhi,danilho doekhi



  1. FSV Mainz 05  —  SF sin salario:


...

,player,player_norm
0,Agon Elezi,agon elezi
1,Andreas Luthe,andreas luthe
2,Anthony Losilla,anthony losilla
3,Bernardo,bernardo
4,Christopher Antwi-Adjei,christopher antwi adjei
5,Cristian Gamboa,cristian gamboa
6,Danilo Soares,danilo soares
7,Erhan Masovic,erhan masovic
8,Felix Passlack,felix passlack
9,Gonçalo Paciência,goncalo paciencia



  VfL Wolfsburg  —  SF sin salario:


,player,minutesPlayed
0,Bennit Bröger,10
1,Kofi Jeremy Amoako,2


  CG plantilla completa:


,player,player_norm
0,Amin Sarr,amin sarr
1,Aster Vranckx,aster vranckx
2,Cédric Zesiger,cedric zesiger
3,Dzenan Pejcinovic,dzenan pejcinovic
4,Felix Lange,felix lange
5,Jakub Kaminski,jakub kaminski
6,Joakim Maehle,joakim maehle
7,Jonas Wind,jonas wind
8,Kevin Behrens,kevin behrens
9,Kevin Paredes,kevin paredes


In [19]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {
    ('silas', 'vfb stuttgart'): ('silas katompa mvumpa', 'vfb stuttgart'),
    ('iago borduchi','fc augsburg'):('iago',('fc augsburg'))
}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')


Matches manuales definidos: 2


In [20]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')

✅ Match manual aplicado: silas (vfb stuttgart) → silas katompa mvumpa (vfb stuttgart)
✅ Match manual aplicado: iago borduchi (fc augsburg) → iago (fc augsburg)

Tras matches manuales: 464/494 (93.9%)


In [21]:
pd.reset_option('display.max_rows')

## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [22]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_germany_2324.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_germany_2324.csv
   Jugadores totales:  494
   Con salario:        464
   Sin salario (NaN):  30
   Columnas:           121
